# 02 — Cleaning and Merge

All 4 datasets are cleaned, merged, and saved as `Data/processed/movies_merged.csv`.
All subsequent analysis and ML notebooks read from that single file.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from Source.scripts.clean_data import (
    add_budget_tier,
    add_decade,
    add_roi_column,
    extract_primary_genre,
    extract_year,
    filter_positive_budget_revenue,
    normalize_title,
)
from Source.scripts.helpers import PROCESSED_DATA_DIR, ensure_processed_dir
from Source.scripts.load_data import load_named_dataset

## Step 1 — movies_metadata: Load and Clean

Rows with zero budget or revenue are meaningless for analysis — they are removed.
Genre is extracted from the JSON string and release year is parsed from the date column.

In [ ]:
meta = load_named_dataset('the_movies_metadata')
print(f'Raw shape: {meta.shape}')

meta = filter_positive_budget_revenue(meta, 'budget', 'revenue')
print(f'After budget/revenue filter: {meta.shape}')

meta['title_clean']   = normalize_title(meta['title'])
meta['release_year']  = extract_year(meta['release_date'])
meta['primary_genre'] = extract_primary_genre(meta['genres'])

keep = [c for c in [
    'id', 'imdb_id', 'title', 'title_clean', 'budget', 'revenue',
    'runtime', 'release_year', 'primary_genre', 'vote_average', 'vote_count', 'popularity',
] if c in meta.columns]
df = meta[keep].copy()

print(f'Working dataset shape: {df.shape}')
df.head(3)

## Step 2 — TMDB: Add tmdb_popularity

TMDB popularity score is added as an additional feature.
Left join on normalized title — unmatched rows stay NaN.

In [ ]:
tmdb = load_named_dataset('tmdb_movies')
tmdb['title_clean'] = normalize_title(tmdb['title'])

if 'popularity' in tmdb.columns:
    tmdb_slim = (
        tmdb[['title_clean', 'popularity']]
        .rename(columns={'popularity': 'tmdb_popularity'})
        .drop_duplicates('title_clean')
    )
    df = df.merge(tmdb_slim, on='title_clean', how='left')
    filled = df['tmdb_popularity'].notna().sum()
    print(f'After TMDB merge: {df.shape} | tmdb_popularity filled: {filled}')
else:
    print('No popularity column found in tmdb_movies')

## Step 3 — IMDb Ratings: Join via imdb_id

Much more reliable than title matching — especially for foreign-language films.
Uses the imdb_id column that already exists in movies_metadata.

In [ ]:
imdb = load_named_dataset('imdb_ratings')
imdb = imdb.rename(columns={
    'tconst': 'imdb_id',
    'averageRating': 'imdb_rating',
    'numVotes': 'imdb_votes',
})

df = df.merge(imdb[['imdb_id', 'imdb_rating', 'imdb_votes']], on='imdb_id', how='left')
print(f'After IMDb merge: {df.shape}')
print(f'imdb_rating filled: {df["imdb_rating"].notna().sum()} / {len(df)}')

## Step 4 — Rotten Tomatoes: Join via Title

For tomatometer and audience rating.
Join is done on normalized titles — some films may not match, which is expected.

In [ ]:
rt = load_named_dataset('rt_movies')
title_col = 'movie_title' if 'movie_title' in rt.columns else 'title'
rt['title_clean'] = normalize_title(rt[title_col])

rt_cols = [c for c in ['title_clean', 'tomatometer_rating', 'audience_rating'] if c in rt.columns]
rt_slim = rt[rt_cols].drop_duplicates('title_clean')

df = df.merge(rt_slim, on='title_clean', how='left')
print(f'After RT merge: {df.shape}')
for col in ['tomatometer_rating', 'audience_rating']:
    if col in df.columns:
        print(f'{col} filled: {df[col].notna().sum()} / {len(df)}')

## Step 5 — Add Derived Columns

ROI, budget tier and decade — used directly in EDA and ML.

In [ ]:
df = add_roi_column(df, 'budget', 'revenue')
df = add_budget_tier(df, 'budget')
df = add_decade(df, 'release_year')

print('budget_tier distribution:')
print(df['budget_tier'].value_counts())
print('\ndecade distribution:')
print(df['decade'].value_counts().sort_index())

## Step 6 — Null Check and Save

In [ ]:
print('Null counts per column:')
print(df.isnull().sum().sort_values(ascending=False))

In [ ]:
ensure_processed_dir()
out_path = PROCESSED_DATA_DIR / 'movies_merged.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows to {out_path}')
df.head(3)